# LAB 02 — AG Binário com Reparação/Penalidade para Seleção de Microsserviços em Edge

Objetivo: maximizar o valor de negócio selecionando microsserviços, respeitando simultaneamente **16 GB de RAM** e **8 cores de CPU**.

Serão comparadas duas estratégias: **A — Penalidade Rígida** e **B — Penalidade Proporcional**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# Dados dos 15 microsserviços
servicos = pd.DataFrame({
    'Servico': [f'S{i+1}' for i in range(15)],
    'Valor': [30, 45, 25, 60, 50, 35, 70, 40, 55, 65, 20, 48, 75, 38, 58],
    'RAM_GB': [2.0, 1.5, 3.0, 2.5, 4.0, 1.0, 3.5, 2.0, 2.5, 4.5, 1.5, 3.0, 4.0, 2.0, 3.5],
    'CPU': [0.8, 0.5, 1.2, 1.0, 1.5, 0.4, 1.8, 0.7, 1.1, 1.6, 0.6, 1.0, 2.0, 0.8, 1.3]
})

MAX_RAM = 16.0
MAX_CPU = 8.0

POP_SIZE = 40
GENERATIONS = 100
TOURNAMENT_SIZE = 3
CROSSOVER_RATE = 0.80
MUTATION_RATE = 0.05

print(servicos.to_string(index=False))


## Funções de avaliação

Na Estratégia A, qualquer violação faz o fitness ser zero. Na Estratégia B, o valor é reduzido proporcionalmente aos excessos de RAM e CPU.

In [ ]:
def avaliar(individuo, estrategia='A'):
    individuo = np.asarray(individuo)
    valor = np.sum(individuo * servicos['Valor'].values)
    ram = np.sum(individuo * servicos['RAM_GB'].values)
    cpu = np.sum(individuo * servicos['CPU'].values)

    excesso_ram = max(0.0, ram - MAX_RAM)
    excesso_cpu = max(0.0, cpu - MAX_CPU)

    if estrategia == 'A':
        if excesso_ram > 0 or excesso_cpu > 0:
            fit = 0.0
        else:
            fit = valor
    else:
        # Penalização proporcional ao percentual excedido.
        penalidade_ram = excesso_ram / MAX_RAM
        penalidade_cpu = excesso_cpu / MAX_CPU
        fator = max(0.0, 1.0 - penalidade_ram - penalidade_cpu)
        fit = valor * fator

    return fit, valor, ram, cpu

def criar_individuo():
    return np.random.randint(0, 2, size=len(servicos))

def selecionar_torneio(populacao, fitnesses):
    indices = np.random.choice(len(populacao), TOURNAMENT_SIZE, replace=False)
    vencedor = indices[np.argmax(fitnesses[indices])]
    return populacao[vencedor].copy()

def crossover_ponto_unico(p1, p2):
    if np.random.rand() > CROSSOVER_RATE:
        return p1.copy(), p2.copy()
    ponto = np.random.randint(1, len(p1))
    return np.concatenate([p1[:ponto], p2[ponto:]]), np.concatenate([p2[:ponto], p1[ponto:]])

def mutacao(individuo):
    mascara = np.random.rand(len(individuo)) < MUTATION_RATE
    individuo[mascara] = 1 - individuo[mascara]
    return individuo


In [ ]:
def executar_ag(estrategia, seed=42):
    np.random.seed(seed)
    populacao = np.array([criar_individuo() for _ in range(POP_SIZE)])
    historico_media = []
    historico_std = []
    historico_diversidade = []
    melhor_individuo = None
    melhor_fitness = -1

    for _ in range(GENERATIONS):
        fitnesses = np.array([avaliar(ind, estrategia)[0] for ind in populacao])

        historico_media.append(np.mean(fitnesses))
        historico_std.append(np.std(fitnesses))
        # Diversidade: proporção média de genes diferentes do gene majoritário.
        frequencias = np.mean(populacao, axis=0)
        diversidade = np.mean(2 * np.minimum(frequencias, 1 - frequencias))
        historico_diversidade.append(diversidade)

        idx = np.argmax(fitnesses)
        if fitnesses[idx] > melhor_fitness:
            melhor_fitness = fitnesses[idx]
            melhor_individuo = populacao[idx].copy()

        nova_pop = []
        # Elitismo: preserva o melhor indivíduo da geração.
        nova_pop.append(populacao[idx].copy())

        while len(nova_pop) < POP_SIZE:
            p1 = selecionar_torneio(populacao, fitnesses)
            p2 = selecionar_torneio(populacao, fitnesses)
            f1, f2 = crossover_ponto_unico(p1, p2)
            nova_pop.append(mutacao(f1))
            if len(nova_pop) < POP_SIZE:
                nova_pop.append(mutacao(f2))

        populacao = np.array(nova_pop)

    return {
        'melhor': melhor_individuo,
        'fitness': melhor_fitness,
        'media': historico_media,
        'std': historico_std,
        'diversidade': historico_diversidade
    }


## Execução das duas estratégias

In [ ]:
resultado_A = executar_ag('A', seed=42)
resultado_B = executar_ag('B', seed=42)

for nome, resultado in [('A — Penalidade Rígida', resultado_A), ('B — Penalidade Proporcional', resultado_B)]:
    ind = resultado['melhor']
    fit, valor, ram, cpu = avaliar(ind, 'A' if nome.startswith('A') else 'B')
    print('\n', nome)
    print('Fitness:', round(fit, 4))
    print('Valor bruto:', round(valor, 4))
    print('RAM:', round(ram, 4), 'GB')
    print('CPU:', round(cpu, 4), 'cores')
    print('Serviços:', [servicos.iloc[i]['Servico'] for i, x in enumerate(ind) if x == 1])


## Média e desvio-padrão do fitness

In [ ]:
geracoes = np.arange(1, GENERATIONS + 1)

plt.figure(figsize=(10, 6))
plt.plot(geracoes, resultado_A['media'], label='Estratégia A')
plt.plot(geracoes, resultado_B['media'], label='Estratégia B')
plt.title('Média do fitness por geração')
plt.xlabel('Geração')
plt.ylabel('Fitness médio')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(geracoes, resultado_A['std'], label='Estratégia A')
plt.plot(geracoes, resultado_B['std'], label='Estratégia B')
plt.title('Desvio-padrão do fitness por geração')
plt.xlabel('Geração')
plt.ylabel('Desvio-padrão')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Diversidade genética

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(geracoes, resultado_A['diversidade'], label='Estratégia A')
plt.plot(geracoes, resultado_B['diversidade'], label='Estratégia B')
plt.title('Diversidade genética da população')
plt.xlabel('Geração')
plt.ylabel('Índice de diversidade')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

comparacao = pd.DataFrame({
    'Métrica': ['Melhor fitness', 'Fitness médio final', 'Desvio-padrão final', 'Diversidade final'],
    'Estratégia A': [resultado_A['fitness'], resultado_A['media'][-1], resultado_A['std'][-1], resultado_A['diversidade'][-1]],
    'Estratégia B': [resultado_B['fitness'], resultado_B['media'][-1], resultado_B['std'][-1], resultado_B['diversidade'][-1]]
})
display(comparacao.round(4))


## Validação da melhor combinação

O resultado final mostra os microsserviços selecionados e confirma os consumos de RAM e CPU. A conclusão sobre diversidade e qualidade deve ser baseada nos valores produzidos pelos gráficos e pela tabela após a execução.

In [ ]:
linhas = []
for nome, resultado, estrategia in [('A', resultado_A, 'A'), ('B', resultado_B, 'B')]:
    ind = resultado['melhor']
    fit, valor, ram, cpu = avaliar(ind, estrategia)
    linhas.append({
        'Estratégia': nome,
        'Fitness': fit,
        'Valor': valor,
        'RAM (GB)': ram,
        'CPU (cores)': cpu,
        'RAM válida': ram <= MAX_RAM,
        'CPU válida': cpu <= MAX_CPU,
        'Qtd. serviços': int(np.sum(ind))
    })

display(pd.DataFrame(linhas).round(4))
